# Explore LoRA internals

Goal: *see* how adaptation actually works, not just run a trainer.

We will:
1. Load a small VLM with a QLoRA config (same code path as the 9B target).
2. Confirm only <1% of parameters are trainable.
3. Look at where adapters are injected (MoE / attention linears).
4. Open one adapted layer and inspect the low-rank matrices `A`, `B`, the scaling `alpha / r`, and the effective update `dW = (alpha/r) * B @ A`.
5. See how `r` and `lora_alpha` change the trainable-param count and update magnitude.

> Run this with the tiny smoke model first. It uses the exact same `build_model` as Qwen3.5-9B, so everything you learn here transfers.

In [ ]:
import sys, pathlib
# Make the src/ package importable from the notebooks/ folder.
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from lora_lab.config import load_run_config

# Use the tiny multimodal smoke config so this loads fast.
cfg = load_run_config(
    ROOT / "configs/model/tiny_mm_smoke.yaml",
    ROOT / "configs/data/cord.yaml",
    ROOT / "configs/train/sft_qlora.yaml",
)
cfg.model.name_or_path, cfg.model.lora

## 1-3. Load model, count trainable params, see injection points

`build_model` loads the 4-bit base, freezes vision, auto-discovers the linear
layers, injects LoRA, and prints the trainable-parameter ratio. Watch the
`print_linear_tree` output: that is literally where adapters get attached.

In [ ]:
from lora_lab.model import build_model

model, processor, targets = build_model(cfg.model, verbose=True)
print("\ntarget_modules:", targets)

## 4. Inspect one adapted layer: A, B, scaling, and dW = (alpha/r) * B @ A

PEFT wraps a target `nn.Linear` in a `lora.Linear` module that keeps the frozen
base weight plus trainable `lora_A` and `lora_B`. The forward pass computes
`W0 @ x + scaling * (B @ (A @ x))`, where `scaling = lora_alpha / r`.

Note `lora_B` is initialized to zeros, so at step 0 `dW = 0` and the adapted
model is identical to the base model - training moves it away from there.

In [ ]:
import torch

# Grab the first LoRA-wrapped layer.
lora_layer = None
lora_name = None
for name, module in model.named_modules():
    if hasattr(module, "lora_A") and "default" in getattr(module, "lora_A", {}):
        lora_layer, lora_name = module, name
        break

print("Adapted layer:", lora_name)
A = lora_layer.lora_A["default"].weight   # shape [r, in_features]
B = lora_layer.lora_B["default"].weight   # shape [out_features, r]
r = lora_layer.r["default"]
alpha = lora_layer.lora_alpha["default"]
scaling = alpha / r

print(f"A: {tuple(A.shape)}  B: {tuple(B.shape)}  r={r}  alpha={alpha}  scaling={scaling}")

# Effective weight update dW has the FULL shape [out, in] but rank <= r.
dW = scaling * (B @ A)
print("dW shape:", tuple(dW.shape), "| rank(dW) <= r =", r)
print("||dW|| at init (B is zero) =", dW.norm().item())

## 5. How r and alpha affect trainable params

For a single adapted linear of shape `[out, in]`, LoRA adds `r * (in + out)`
trainable parameters instead of `in * out`. Below we compute the savings and
the effect of changing `r`.

In [ ]:
out_features, in_features = B.shape[0], A.shape[1]
full = in_features * out_features
print(f"Layer [{out_features} x {in_features}] full weight params: {full:,}\n")
print(f"{'r':>5} {'lora_params':>14} {'% of full':>10}")
for rank in (4, 8, 16, 32, 64):
    lp = rank * (in_features + out_features)
    print(f"{rank:>5} {lp:>14,} {100*lp/full:>9.2f}%")

# alpha does NOT change the parameter count - only the update SCALE (alpha/r).
print("\nScaling factor alpha/r controls how strongly dW is applied, independent of param count.")

## Takeaways

- LoRA freezes `W0` and learns a low-rank `dW = (alpha/r) * B @ A`; only `A`, `B` train.
- QLoRA additionally stores `W0` in 4-bit NF4, so a 10B model fits on a 32 GB 5090.
- `r` trades capacity for memory; `alpha/r` is the update scale. Common start: `r=16, alpha=32`.
- `B` starts at zero => the adapted model begins identical to the base, then diverges during training.
- The same `build_model` path scales from this tiny model to `Qwen/Qwen3.5-9B` by swapping the model YAML.

Next: run `scripts/02_train.py --smoke` to train a few steps, then
`scripts/03_evaluate.py` to compare baseline vs adapter.